# Hybrid QKD + ML-KEM Wrapping
### The PQC store-and-forward companion to the BB84 QBER notebook

**Pairs with:** `bb84_qkd_eavesdropping_ibm_quantum.ipynb`
**Environment:** Any Python 3.10+ (qBraid default kernel works; no quantum hardware needed)
**Runtime:** < 5 seconds, all classical
**Result:** A hardware-realistic demonstration that the bank can wrap a sensitive payload using a session key derived from *both* a QKD-derived shared secret **and** an ML-KEM (FIPS 203) encapsulation — the construction NIST, BSI, and NSA all recommend for the post-quantum transition period.

## Why this notebook exists

The BB84 notebook demonstrates the **real-time channel** leg of post-quantum security: Alice and Bob agree on a shared key over a quantum channel, with eavesdropping detection via QBER. But a real bank doesn't only care about real-time channels — it also has to:

- archive payloads for regulatory retention,
- forward transactions through correspondent networks,
- replicate data to backup sites,

…all of which involve cryptographic operations on data that won't be touched again for years. The threat model there isn't "is the channel being tapped right now" — it's "harvest now, decrypt later": an adversary captures ciphertext today, stores it, and decrypts it once a cryptographically relevant quantum computer becomes available. That's the **store-and-forward** leg, and QKD alone doesn't solve it. PQC does.

This notebook stitches the two together. The pattern is exactly the **hybrid KEM** construction that Cloudflare, AWS, and Signal have deployed at internet scale since 2023 — except here the "classical" component is replaced by QKD, giving you a triple guarantee: secure if *either* QKD *or* ML-KEM holds, plus authenticated tampering detection on the ciphertext via AES-256-GCM.

## Protocol architecture

```
   ALICE (sender side)                                BOB (recipient side)
   ─────────────────────                              ────────────────────

   ┌─────────────────────────────────────────────────────────────────┐
   │   1.  QKD exchange (BB84 notebook)                              │
   │       Outputs:  K_QKD   (32 bytes, sifted+privacy-amplified)    │
   └─────────────────────────────────────────────────────────────────┘
            │                                                  │
            │            K_QKD                          K_QKD  │
            │                                                  │
            │              ←── ek_bob (ML-KEM public key) ──   │  2. one-time PQC setup
            │                                                  │
   3. ML-KEM encapsulation                                     │
      (K_Kyber, ct) = ML_KEM_1024.encaps(ek_bob)               │
            │                                                  │
   4. Hybrid combine                                           │
      K_final = HKDF-SHA256( K_QKD || K_Kyber , salt, info )   │
            │                                                  │
   5. AES-256-GCM wrap                                         │
      (nonce, wrapped) = AESGCM(K_final).encrypt(payload, AAD) │
            │                                                  │
            │              ── ct, nonce, wrapped, AAD ──→     │
            │                                                  │
            │                                       6. K_Kyber' = ML_KEM_1024.decaps(dk_bob, ct)
            │                                                  │
            │                                       7. K_final' = HKDF-SHA256(K_QKD || K_Kyber', salt, info)
            │                                                  │
            │                                       8. payload = AESGCM(K_final').decrypt(nonce, wrapped, AAD)
            │                                                  │
   ════════════════════════════════════════════════════════════════════
```

**Why the construction is secure.** The hybrid security argument (Bindel et al. 2019, Aviram et al. 2020) shows that an HKDF combiner over the concatenation `K_QKD || K_Kyber` produces a session key indistinguishable from random *as long as at least one* of `K_QKD` or `K_Kyber` is indistinguishable from random to the adversary. So:

- if QKD turns out to have implementation flaws (most likely culprit in practice), PQC still protects you;
- if ML-KEM is broken by future cryptanalysis (unlikely but non-zero), QKD still protects you;
- both being broken simultaneously is the only path to compromise.

## 1. Imports and dependencies

The only non-standard dependency is `kyber-py` (pure-Python ML-KEM, FIPS 203). `cryptography` is bundled in most scientific Python environments including qBraid's Qiskit kernel.

In [ ]:
 %pip install -q kyber-py cryptography

In [ ]:
import os
import json
import time
from binascii import hexlify

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

from kyber_py.ml_kem import ML_KEM_1024  # NIST security level V (~256-bit classical)

print("All imports OK")
print(f"Using ML-KEM-1024 (FIPS 203, NIST security level V)")

## 2. Acquire `K_QKD` — the QKD-derived shared secret

In production, `K_QKD` would be the **privacy-amplified output** of a QKD run — the bits that remain after sifting matching-basis rounds, dropping the test subset used for QBER estimation, doing error reconciliation (e.g. Cascade or LDPC), and applying privacy amplification via a 2-universal hash to compress out any information Eve might have gleaned.

For this notebook we provide two code paths:

- **Path A (demo):** generate `K_QKD` from a CSPRNG. The cryptographic API surface below is identical to the production case.
- **Path B (real QKD output):** paste in a hex string of the actual sifted+amplified bits from the BB84 notebook. Commented out below; uncomment when you have real output.

Either way `K_QKD` ends up as 32 bytes (256 bits).

In [ ]:
# --- Path A: demo (CSPRNG) ---
K_QKD = os.urandom(32)

# --- Path B: real BB84 output (uncomment and replace) ---
# K_QKD_hex = "a1b2c3...  (paste 64 hex chars from the BB84 notebook's sifted key)"
# assert len(K_QKD_hex) == 64
# K_QKD = bytes.fromhex(K_QKD_hex)

assert len(K_QKD) == 32, "K_QKD must be 32 bytes (256 bits)"
print(f"K_QKD  : {hexlify(K_QKD).decode()}")
print(f"  length: {len(K_QKD)} bytes = {len(K_QKD)*8} bits")

## 3. Bob's ML-KEM-1024 keypair (one-time setup)

Bob generates an ML-KEM-1024 keypair once and publishes the encapsulation key (`ek_bob`) — analogous to an X.509 certificate. The decapsulation key (`dk_bob`) is held in Bob's HSM or key vault.

**Parameter set choice.** ML-KEM-1024 gives NIST security level V (~256-bit classical, ~128-bit quantum), matching AES-256. NIST recommends ML-KEM-768 (level III) as a default; we use 1024 here because banking workloads typically have 25+ year retention requirements and need the largest practical security margin against future cryptanalytic progress.

In [ ]:
t0 = time.perf_counter()
ek_bob, dk_bob = ML_KEM_1024.keygen()
keygen_ms = (time.perf_counter() - t0) * 1000

print(f"ek_bob (encapsulation key, public)  : {len(ek_bob):>5} bytes")
print(f"dk_bob (decapsulation key, private) : {len(dk_bob):>5} bytes")
print(f"keygen time                         : {keygen_ms:6.2f} ms")

## 4. Alice encapsulates against Bob's public key

`ML_KEM_1024.encaps(ek_bob)` returns:

- `K_Kyber` — a 32-byte shared secret that **only Bob can recover** (using `dk_bob`),
- `ct` — a ciphertext that Alice transmits to Bob.

`K_Kyber` is the PQC half of the hybrid session key. It's never sent over the wire; only `ct` is.

In [ ]:
t0 = time.perf_counter()
K_Kyber_alice, ct = ML_KEM_1024.encaps(ek_bob)
encaps_ms = (time.perf_counter() - t0) * 1000

print(f"K_Kyber (sender's view)  : {hexlify(K_Kyber_alice).decode()}")
print(f"  length                : {len(K_Kyber_alice)} bytes")
print(f"ct (transmitted to Bob)  : {len(ct)} bytes")
print(f"encaps time              : {encaps_ms:6.2f} ms")

## 5. Hybrid key combination via HKDF-SHA256

Both inputs feed into an HKDF combiner ([RFC 5869](https://datatracker.ietf.org/doc/html/rfc5869)). The construction follows [NIST SP 800-56C Rev. 2](https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-56Cr2.pdf) for hybrid key derivation.

- **IKM** (input keying material) = `K_QKD || K_Kyber`  — 64 bytes total
- **salt** — domain separator identifying this specific protocol and version. Public, not secret.
- **info** — purpose label for the derived key; lets the same IKM produce different keys for different operations (e.g. encryption vs. MAC).
- **L** = 32 — output length, matching the AES-256 key size.

The HKDF combiner is what guarantees the "secure if either input is secure" property. Concatenation alone would not — a properly designed KDF is required to prevent an adversary from substituting one half.

In [ ]:
def hybrid_kdf(k_qkd: bytes, k_pqc: bytes, salt: bytes, info: bytes, length: int = 32) -> bytes:
    """NIST SP 800-56C Rev. 2 compliant hybrid key combiner."""
    ikm = k_qkd + k_pqc
    return HKDF(
        algorithm=hashes.SHA256(),
        length=length,
        salt=salt,
        info=info,
    ).derive(ikm)

SALT = b"cibc-hybrid-qkd-mlkem-v1"
INFO = b"AES-256-GCM-wrapping-key"

K_final_alice = hybrid_kdf(K_QKD, K_Kyber_alice, SALT, INFO)
print(f"K_final (Alice)  : {hexlify(K_final_alice).decode()}")
print(f"  length        : {len(K_final_alice)} bytes  (AES-256 key)")

## 6. Wrap a sensitive payload with AES-256-GCM

The session key `K_final` is fed into AES-256-GCM, which provides:

- **Confidentiality** — the payload is encrypted under a key the adversary cannot derive.
- **Integrity** — any tampering with the ciphertext or AAD invalidates the GCM authentication tag.
- **Non-malleability** — the AAD ("additional authenticated data") binds metadata such as sender, recipient, and classification to the ciphertext, preventing misrouting attacks.

Sample payload: a synthetic BiMPay-style instant payment record. In practice this could be a SWIFT message batch, a customer KYC bundle, a regulatory submission, or a backup snapshot.

In [ ]:
payload = json.dumps({
    "scheme": "BiMPay",
    "transaction_id": "BMP-2026-05-20-0000123456",
    "timestamp_utc": "2026-05-20T14:32:11Z",
    "sender":   {"account": "BB...8821", "bank": "FCIB"},
    "receiver": {"account": "BB...4407", "bank": "RBC"},
    "amount":   {"value": "1450.00", "currency": "BBD"},
    "narrative": "Q2 supplier invoice INV-2026-0455",
    "compliance": {"sanctions_check": "pass", "aml_score": 0.12},
}).encode("utf-8")

# AAD binds metadata to the ciphertext (not encrypted, but authenticated)
AAD = b"channel=hybrid-qkd-mlkem-v1;sender=sender@example.com;classification=confidential"

# 96-bit nonce per NIST SP 800-38D
nonce = os.urandom(12)

aesgcm_alice = AESGCM(K_final_alice)
wrapped = aesgcm_alice.encrypt(nonce, payload, AAD)

print(f"payload (plaintext)   : {len(payload)} bytes")
print(f"wrapped (ciphertext)  : {len(wrapped)} bytes   ({len(wrapped) - len(payload)} bytes of GCM tag overhead)")
print(f"nonce                 : {hexlify(nonce).decode()}")
print()
print("Transmitted to Bob: (ct, nonce, wrapped, AAD)")

## 7. Bob's side: decapsulate, recombine, unwrap

Bob receives `(ct, nonce, wrapped, AAD)` from Alice. He already has `K_QKD` from the BB84 run and `dk_bob` from his keypair. He runs the inverse pipeline:

1. `ML_KEM_1024.decaps(dk_bob, ct)` → `K_Kyber` (must match Alice's)
2. `HKDF(K_QKD || K_Kyber, ...)` → `K_final` (must match Alice's)
3. `AESGCM(K_final).decrypt(...)` → original `payload`

Any failure at any step (wrong key, tampered ciphertext, tampered AAD) raises `InvalidTag` and Bob aborts.

In [ ]:
# --- Step 6a: ML-KEM decapsulation ---
t0 = time.perf_counter()
K_Kyber_bob = ML_KEM_1024.decaps(dk_bob, ct)
decaps_ms = (time.perf_counter() - t0) * 1000
print(f"K_Kyber (Bob's view)  : {hexlify(K_Kyber_bob).decode()}")
print(f"decaps time           : {decaps_ms:6.2f} ms")
print(f"K_Kyber_alice == K_Kyber_bob ? {K_Kyber_alice == K_Kyber_bob}")
print()

# --- Step 6b: Hybrid key recombination ---
K_final_bob = hybrid_kdf(K_QKD, K_Kyber_bob, SALT, INFO)
print(f"K_final (Bob)         : {hexlify(K_final_bob).decode()}")
print(f"K_final matches?      : {K_final_alice == K_final_bob}")
print()

# --- Step 6c: AES-GCM unwrap ---
aesgcm_bob = AESGCM(K_final_bob)
recovered = aesgcm_bob.decrypt(nonce, wrapped, AAD)
print(f"recovered payload     : {len(recovered)} bytes")

## 8. Verify and tamper-test

Two checks: that the recovered payload exactly matches the original, and that *any* tampering with the ciphertext or AAD is detected.

In [ ]:
# --- Integrity ---
assert recovered == payload, "recovered payload does not match original!"
print("✓ Payload recovered byte-for-byte\n")

# --- Tamper test 1: flip one bit in the ciphertext ---
from cryptography.exceptions import InvalidTag

tampered_ct = bytearray(wrapped)
tampered_ct[0] ^= 0x01
try:
    aesgcm_bob.decrypt(nonce, bytes(tampered_ct), AAD)
    print("✗ FAILED: tampered ciphertext was accepted")
except InvalidTag:
    print("✓ Tampered ciphertext correctly rejected (InvalidTag)")

# --- Tamper test 2: modify the AAD (e.g. attacker rewrites 'classification') ---
tampered_aad = AAD.replace(b"confidential", b"public")
try:
    aesgcm_bob.decrypt(nonce, wrapped, tampered_aad)
    print("✗ FAILED: tampered AAD was accepted")
except InvalidTag:
    print("✓ Tampered AAD correctly rejected (InvalidTag)")

# --- Tamper test 3: substitute a different Kyber ciphertext (downgrade attempt) ---
_, fake_ct = ML_KEM_1024.encaps(ek_bob)  # different encapsulation
K_Kyber_fake = ML_KEM_1024.decaps(dk_bob, fake_ct)
K_final_fake = hybrid_kdf(K_QKD, K_Kyber_fake, SALT, INFO)
try:
    AESGCM(K_final_fake).decrypt(nonce, wrapped, AAD)
    print("✗ FAILED: substituted Kyber ciphertext was accepted")
except InvalidTag:
    print("✓ Substituted Kyber ciphertext correctly rejected (InvalidTag)")

## 9. Artifact summary

Numbers a CISO or board briefing would want to cite directly.

In [ ]:
print("ARTIFACT SIZES")
print("=" * 60)
print(f"  ek_bob   (ML-KEM-1024 public)        : {len(ek_bob):>5} bytes")
print(f"  dk_bob   (ML-KEM-1024 private)       : {len(dk_bob):>5} bytes")
print(f"  ct       (KEM ciphertext, per session): {len(ct):>5} bytes")
print(f"  K_QKD    (QKD shared secret)         : {len(K_QKD):>5} bytes (256 bits)")
print(f"  K_Kyber  (KEM shared secret)         : {len(K_Kyber_alice):>5} bytes (256 bits)")
print(f"  K_final  (AES-256 session key)       : {len(K_final_alice):>5} bytes (256 bits)")
print(f"  nonce    (AES-GCM)                   : {len(nonce):>5} bytes ( 96 bits)")
print(f"  GCM tag overhead                     : {len(wrapped) - len(payload):>5} bytes (128 bits)")
print()
print("PER-SESSION WIRE OVERHEAD")
print("=" * 60)
overhead = len(ct) + len(nonce) + (len(wrapped) - len(payload))
print(f"  ct + nonce + tag                     : {overhead:>5} bytes  (~{overhead/1024:.2f} KB)")
print()
print("TIMING (this notebook's run)")
print("=" * 60)
print(f"  ML-KEM keygen                        : {keygen_ms:7.2f} ms")
print(f"  ML-KEM encaps                        : {encaps_ms:7.2f} ms")
print(f"  ML-KEM decaps                        : {decaps_ms:7.2f} ms")
print()
print("SECURITY CLAIMS")
print("=" * 60)
print("  Classical security level             : ~256-bit (ML-KEM-1024 + AES-256)")
print("  Quantum security level               : ~128-bit (Grover-bounded)")
print("  Standards compliance                 : FIPS 203, NIST SP 800-56C Rev. 2, NIST SP 800-38D")
print("  Hybrid guarantee                     : Secure if EITHER QKD OR ML-KEM holds")
print("  Tamper detection                     : AES-256-GCM (128-bit auth tag)")

## 10. Mapping to standards and policy guidance

| Standard / Guidance | What this notebook satisfies |
|---|---|
| **NIST FIPS 203** (ML-KEM) | Uses ML-KEM-1024 keygen/encaps/decaps as specified |
| **NIST SP 800-56C Rev. 2** | Hybrid key derivation via HKDF-SHA256 over concatenated IKM |
| **NIST SP 800-38D** | AES-256-GCM with 96-bit nonce and 128-bit auth tag |
| **BSI TR-02102-1** | Hybrid PQC+classical (here PQC+QKD) for transition period |
| **NSA CNSA 2.0** | ML-KEM-1024 listed as approved for NSS by 2035 deadline |
| **ETSI TS 119 312** | Long-term protection profile (≥ 25 years) — matches banking retention |

## 11. Limitations and production notes

**`kyber-py` is for demonstration, not production.** It's pure Python — significantly slower than C/Rust implementations (~60–70× slower per the `mlkem` package benchmarks) and has not been audited or side-channel-hardened. For production:

- **liboqs** (Open Quantum Safe) — C library with Python bindings; audited; used by Cloudflare, AWS.
- **OpenSSL 3.5+** — includes ML-KEM-512/768/1024 as named groups.
- **PyCA `cryptography`** — likely to add ML-KEM in a near-future release; track [PR #11367](https://github.com/pyca/cryptography).
- **HSM integration** — production deployments should hold `dk_bob` in an HSM that supports PQC (Thales Luna, Entrust nShield, AWS CloudHSM all have ML-KEM roadmaps).

**This notebook doesn't address:**

- **Authentication of `ek_bob`.** Bob's encapsulation key needs to be authenticated (PKI certificate, out-of-band fingerprint, or pre-shared trust). Without that, Eve can mount a classical MITM by substituting her own `ek`. In production this is handled by a PQC certificate (e.g. ML-DSA / Dilithium signature).
- **Key rotation and forward secrecy.** A real deployment rotates `ek_bob` and the QKD session keys on a schedule. The hybrid construction here is one-shot.
- **Side-channel resistance.** The pure-Python `kyber-py` has no constant-time guarantees. ML-KEM is known to be vulnerable to timing attacks on naive implementations — see the Hertzbleed and KyberSlash papers (2022–2024).
- **Quantum random number generation.** Both `K_QKD` (in Path A) and `dk_bob` should ultimately be seeded from a QRNG in production. ID Quantique Quantis or IBM's `qrng` API are common choices.

## 12. Next steps for the CIBC quantum-readiness narrative

This notebook closes the loop on the bank's PQ transition story by demonstrating that:

1. **Quantum-secure channels are testable today** (BB84 notebook): the bank has hands-on competence with the primitive every commercial QKD vendor sells.
2. **Quantum-safe storage is deployable today** (this notebook): the bank can wrap any payload with FIPS-approved post-quantum primitives.
3. **Hybrid is the responsible path** (the combination): protection against both classical adversaries and harvest-now-decrypt-later attacks, with redundancy if either leg fails.

For board or regulator briefings, the two notebooks together produce four artifacts worth attaching:

- The BB84 detection chart (QBER bars with thresholds) — evidence of real-time channel security
- The CHSH or BB84 noise-floor numbers — evidence of empirical hardware calibration
- The hybrid key combination diagram (Section 1 of this notebook) — the architecture
- The artifact size summary (Section 9) — operational footprint numbers

Together they make the case that CIBC Caribbean has practitioner-level competence across both legs of the post-quantum transition — not vendor-marketing competence.